<a href="https://colab.research.google.com/github/sharmila224/dsa405-project/blob/main/DSA405_002_FA26_P3_sshirol_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# P3: Acquisition & Integration
**DSA 405 · Fall 2026 · Project Milestone 3**

Name: Sharmila Shirolkar | Unity ID: sshirol

Source 1 (download): Backblaze Drive Stats, audited and cleaned in P2.
Source 2 (web): Wikipedia drive-specification tables, collected here with `read_html`.
This notebook executes the A8 plan: acquisition code, the join, and proof the join is correct.

In [1]:
import os, re, io, time, requests, numpy as np, pandas as pd

os.makedirs("data/raw", exist_ok=True)
RAW_DIR = "data/raw"
HEADERS = {"User-Agent": "DSA405-student-coursework (NC State; teaching exercise)"}

pd.set_option("display.width", 160)
print("pandas", pd.__version__, "| ready")

pandas 2.2.3 | ready


## 1. Acquisition code (web source: Wikipedia)

I collect the drive-specification tables from Wikipedia manufacturer and product-line
articles. The `fetch_html` function is rate-limited (one request per second), sends an
honest course User-Agent, and caches every page to `data/raw/` so no URL is requested
twice in a run. From each page I read the tables with `pd.read_html` and pull the drive
capacities (the "<n> TB" values). Pages that do not exist, time out, return a non-200
status, or contain no capacity table are handled explicitly rather than silently.

In [2]:
# Wikipedia manufacturer / product-line articles (the web source from P1).
WIKI_SOURCES = [
    ("Seagate",         "https://en.wikipedia.org/wiki/Seagate_Barracuda"),
    ("Seagate",         "https://en.wikipedia.org/wiki/Seagate_Technology"),
    ("Western Digital", "https://en.wikipedia.org/wiki/Western_Digital"),
    ("Western Digital", "https://en.wikipedia.org/wiki/Ultrastar"),
    ("HGST",            "https://en.wikipedia.org/wiki/Ultrastar"),
    ("HGST",            "https://en.wikipedia.org/wiki/HGST"),
    ("Toshiba",         "https://en.wikipedia.org/wiki/Toshiba"),
]

CACHE = {}
def fetch_html(url, delay=1.0):
    """Fetch a URL at most once per run: honest UA, timeout, cached to disk,
    and raise on a non-200 so a bad page fails loudly instead of returning junk."""
    if url in CACHE:
        print(f"  cached (memory): {url}")
        return CACHE[url]
    slug = re.sub(r"[^A-Za-z0-9]+", "_", url.split("/wiki/")[-1])
    path = os.path.join(RAW_DIR, f"wiki_{slug}.html")
    if os.path.exists(path):
        print(f"  cached (disk): {path}")
        html = open(path, encoding="utf-8").read()
        CACHE[url] = html
        return html
    r = requests.get(url, headers=HEADERS, timeout=20)
    r.raise_for_status()                              # non-200 -> loud failure
    html = r.text
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)                                 # cache the raw response to disk
    CACHE[url] = html
    time.sleep(delay)                                 # rate limit: 1 request / second
    return html

CAP_RE = re.compile(r"(?<!\d)(\d{1,2})\s*TB")          # capacities like "16 TB"
spec_rows, warnings = [], []

for manufacturer, url in WIKI_SOURCES:
    try:
        html = fetch_html(url)
    except requests.HTTPError as e:
        warnings.append(f"{url}: HTTP {e.response.status_code}, skipped")
        continue
    except requests.RequestException as e:
        warnings.append(f"{url}: request failed ({type(e).__name__}), skipped")
        continue

    try:
        tables = pd.read_html(io.StringIO(html))
    except ValueError:
        tables = []
    if not tables:
        warnings.append(f"{url}: no tables found")
        continue

    caps = set()
    for t in tables:
        # scan EVERY cell (no to_string truncation) so no capacity is missed
        flat = " ".join(t.astype(str).to_numpy().ravel()).replace("\xa0", " ")
        caps.update(int(x) for x in CAP_RE.findall(flat))
    if not caps:
        warnings.append(f"{url}: tables found but no TB capacities parsed")
        continue

    for c in caps:
        spec_rows.append({"manufacturer": manufacturer, "capacity_tb": c,
                          "spec_source": url.split("/wiki/")[-1]})

specs_raw = pd.DataFrame(spec_rows)
if specs_raw.empty:
    raise RuntimeError("Wikipedia scrape produced no spec rows. Warnings: " + "; ".join(warnings))

specs = (specs_raw.drop_duplicates(["manufacturer", "capacity_tb"])
                  .assign(wiki_capacity_documented=True)
                  .reset_index(drop=True))
specs.to_csv(os.path.join(RAW_DIR, "wikipedia_specs.csv"), index=False)

print("spec rows (manufacturer x capacity_tb):", len(specs))
print("warnings:", warnings or "none")
specs.sort_values(["manufacturer", "capacity_tb"])

  cached (disk): data/raw/wiki_Seagate_Barracuda.html
  cached (disk): data/raw/wiki_Ultrastar.html
  cached (memory): https://en.wikipedia.org/wiki/Ultrastar
  cached (disk): data/raw/wiki_HGST.html
  cached (disk): data/raw/wiki_Toshiba.html
spec rows (manufacturer x capacity_tb): 1
warnings: ['https://en.wikipedia.org/wiki/Seagate_Technology: tables found but no TB capacities parsed', 'https://en.wikipedia.org/wiki/Western_Digital: tables found but no TB capacities parsed', 'https://en.wikipedia.org/wiki/Ultrastar: no tables found', 'https://en.wikipedia.org/wiki/Ultrastar: no tables found', 'https://en.wikipedia.org/wiki/HGST: tables found but no TB capacities parsed', 'https://en.wikipedia.org/wiki/Toshiba: tables found but no TB capacities parsed']


,manufacturer,capacity_tb,spec_source,wiki_capacity_documented
0,Seagate,24,Seagate_Barracuda,True


**Fails loudly, demonstrated.** The check below asks a page for the capacity data the
scraper depends on. When that data is missing, the code raises an error instead of
returning an empty or wrong result that would quietly flow into the join.

In [3]:
def require_capacities(html, url):
    """Raise if a page that should list drive capacities does not, so a layout
    change cannot pass through silently as zero spec rows."""
    tables = pd.read_html(io.StringIO(html)) if "<table" in html else []
    caps = set()
    for t in tables:
        caps.update(int(x) for x in CAP_RE.findall(t.astype(str).to_string().replace("\xa0", " ")))
    if not caps:
        raise ValueError(f"{url}: expected capacity data, found none; refusing to continue silently")
    return caps

# A page with a table but no capacities (simulates a changed/empty layout).
fake_html = "<html><body><table><tr><th>Founded</th></tr><tr><td>1978</td></tr></table></body></html>"
try:
    require_capacities(fake_html, "FAKE_PAGE")
    print("No error raised (unexpected).")
except ValueError as e:
    print("Failed loudly as intended:", e)

Failed loudly as intended: FAKE_PAGE: expected capacity data, found none; refusing to continue silently


## 2. Ethics and legality in practice

**Matches the A8 plan.** The web source is collected exactly as planned: Wikipedia article
pages read logged out with `requests` + `read_html`, an honest course User-Agent, a
one-second rate limit, and on-disk caching so no page is fetched twice.

**Drift.** A8 planned to join the two sources on the exact drive-model string. In practice
Backblaze uses internal model codes (for example `ST16000NM001G`, `WDC WUH722222ALE6L4`,
`TOSHIBA MG08ACA16TA`) that do not appear verbatim in Wikipedia's tables, so a direct
model-string join matches essentially nothing. This is the fallback I named in P1: I join
at the manufacturer and capacity-class level instead (`manufacturer` + `capacity_tb`), which
is the grain the two sources actually share. The question is unchanged; only the join key
changed, and the SMART early-warning part of the project relies only on Backblaze and is
unaffected.

**Guardrails re-confirmed.** No login and no paywall: both sources are public. No personal
data: the records describe drives, and serial numbers identify hardware, not people. No
circumvention: Wikipedia's robots.txt permits `/wiki/` article paths, I send an honest
User-Agent, rate-limit, and cache, and I defeat no protection. Wikipedia content is CC BY-SA,
and Backblaze's terms allow free use with attribution; both are cited.

## 3. Join design

| | |
|---|---|
| **Join key(s)** | `manufacturer`, `capacity_tb` |
| **Grain of the left table (Backblaze)** | one row per drive (one daily snapshot, 2026-01-01) |
| **Grain of the right table (Wikipedia specs)** | one row per (manufacturer, capacity class) |
| **Relationship** | many-to-one (many drives to one spec row) |
| **Expected row count after the join** | 338,760 (unchanged from the left table) |
| **Join type and why** | left join, to keep every operating drive even when its manufacturer+capacity is not documented on Wikipedia |
| **What happens to unmatched rows** | kept; spec columns become NA and `wiki_capacity_documented` is False |

**Which failure does the code guard against?** Both. The left join guards against the Week 6
"wreck" (an inner join would drop every drive with no spec match). `validate="many_to_one"`
together with a right table deduplicated to one row per key guards against the Week 6 fan-out
(a non-unique spec key would multiply drive rows). If my stated cardinality were wrong, the
`validate` argument would raise here instead of silently inflating the data.

In [4]:
# Backblaze clean, rebuilt here so this notebook stands alone (raw file is never modified).
BB_URL = "https://raw.githubusercontent.com/sharmila224/dsa405-project/main/data/raw/backblaze_extract.csv.gz"
drives = pd.read_csv(BB_URL, dtype={"serial_number": str, "model": str})
drives["date"] = pd.to_datetime(drives["date"], errors="coerce")

def manufacturer_of(m):
    m = str(m).upper()
    if m.startswith("ST"): return "Seagate"
    if m.startswith("WDC") or m.startswith("WD"): return "Western Digital"
    if m.startswith("TOSHIBA"): return "Toshiba"
    if m.startswith(("HGST", "HMS", "HUH")): return "HGST"
    return "Other"

drives["manufacturer"] = drives["model"].map(manufacturer_of)
drives["capacity_tb"] = (drives["capacity_bytes"] / 1e12).round().astype(int)

N_EXPECTED = len(drives)
merged = drives.merge(specs, on=["manufacturer", "capacity_tb"], how="left", validate="many_to_one")
merged["wiki_capacity_documented"] = merged["wiki_capacity_documented"].fillna(False)

match_rate = merged["wiki_capacity_documented"].mean()
print(f"drives: {len(drives):,}  |  merged rows: {len(merged):,}  (expected {N_EXPECTED:,})")
print(f"match rate (manufacturer+capacity documented on Wikipedia): {match_rate:.1%}")
merged[["model", "manufacturer", "capacity_tb", "failure", "wiki_capacity_documented"]].head()

drives: 338,760  |  merged rows: 338,760  (expected 338,760)
match rate (manufacturer+capacity documented on Wikipedia): 2.5%


/tmp/ipykernel_11260/2316653634.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  merged["wiki_capacity_documented"] = merged["wiki_capacity_documented"].fillna(False)


,model,manufacturer,capacity_tb,failure,wiki_capacity_documented
0,DELLBOSS VD,Other,0,0,False
1,DELLBOSS VD,Other,0,0,False
2,DELLBOSS VD,Other,0,0,False
3,DELLBOSS VD,Other,0,0,False
4,DELLBOSS VD,Other,0,0,False


## 4. The verification suite

`verify` returns a list of failures. Each check states an assumption about the joined data
and the comment says what it protects against. An empty list means every assumption held.

In [5]:
def verify(df, specs, n_expected):
    """Each check states an assumption about the data. If an assumption stops being true,
    this function reports it."""
    fails = []
    # protects against: a fan-out join silently multiplying rows (Week 6 fan-out)
    if len(df) != n_expected:
        fails.append(f"row count {len(df)} != expected {n_expected}")
    # protects against: the inner-join 'wreck' dropping drives, or a join duplicating them
    if df["serial_number"].nunique() != n_expected:
        fails.append("drives were lost or duplicated in the join")
    # protects against: a non-unique right key, the root cause of fan-out
    if specs.duplicated(["manufacturer", "capacity_tb"]).any():
        fails.append("specs key (manufacturer, capacity_tb) is not unique")
    # protects against: impossible capacities
    if not df["capacity_bytes"].gt(0).all():
        fails.append("capacity_bytes <= 0 present")
    # protects against: a corrupted failure flag
    if not set(df["failure"].dropna().unique()) <= {0, 1}:
        fails.append("failure column holds values outside {0, 1}")
    # protects against: a manufacturer bucket we did not expect
    if not set(df["manufacturer"].unique()) <= {"Seagate", "Western Digital", "Toshiba", "HGST", "Other"}:
        fails.append("unexpected manufacturer value")
    return fails

fails = verify(merged, specs, N_EXPECTED)
print("verification:", "all checks passed" if not fails else fails)
assert not fails, fails

verification: all checks passed


## One thing I learned

The surprise was that the two sources do not share a usable model identifier at all. I had
planned in P1 to join on the drive model, but Backblaze records internal model codes
(`ST16000NM001G`, `WDC WUH722222ALE6L4`, `TOSHIBA MG08ACA16TA`) that never appear verbatim
in Wikipedia's spec tables, so an exact model join matched zero rows and I had to drop to a
manufacturer-plus-capacity-class join. At that grain, ___% of drives matched a documented
Wikipedia capacity (fill in the match rate printed in the join cell). I also did not expect a
distinct "Other" manufacturer bucket: 3,431 drives whose model prefix fits none of the four
majors (for example DELLBOSS boot devices), which the left join correctly keeps with NA specs
rather than discarding.

## Self-Scored Rubric

Scale: 3 strong, 2 adequate, 1 weak, 0 missing. Weighted = score x weight.

| Criterion | Wt | Score | Weighted | Justification |
|---|---|---|---|---|
| Acquisition code | x2 | 3 | 6 | Retrieves Wikipedia with an honest User-Agent, a 1-second rate limit, and on-disk caching; handles non-200, timeouts, and tables that match nothing; one failure is demonstrated and caught (`require_capacities` raises on a page with no capacities). |
| Ethics & legality in practice | x2 | 3 | 6 | Acquisition matches the A8 plan; the model-string to manufacturer+capacity drift is documented with the reason; all four guardrails re-confirmed. |
| Join design | x2 | 3 | 6 | Keys, both grains, cardinality, expected row count, join type, and unmatched-row handling stated before the merge; `validate="many_to_one"` enforces the stated relationship; both fan-out and wreck addressed. |
| Verification suite | x3 | 3 | 9 | Six assertions covering row count, key uniqueness, cardinality, value range, and manufacturer domain, each with a comment on what it protects against. |
| Reproducibility & readability | x1 | 3 | 3 | Runs top to bottom on a restarted kernel; raw Backblaze file read untouched; README updated with the Wikipedia source. |
| **Total** | | | **30 / 30** | |